In [ ]:
import pandas as pd
import string
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertModel, BertTokenizer

In [131]:
CO_FILE_PATH = "Book2_ELECTIVE4.csv"
PO_FILE_PATH = "Book1_ELECTIVE4.csv"
HUMAN_MAPPING_FILE = "CO_POMapping_ELECTIVE4.csv"

In [132]:
po_data = pd.read_csv(PO_FILE_PATH)
co_data = pd.read_csv(CO_FILE_PATH)

In [133]:
def preprocess_text(text):
    """Converts text to lowercase and removes punctuation."""
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    return text

In [134]:
po_data['cleaned_PO_Description'] = po_data['PO_Description'].apply(preprocess_text)
co_data['cleaned_CO_Description'] = co_data['CO_Description'].apply(preprocess_text)

In [135]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

print("BERT tokenizer and model loaded successfully.")

BERT tokenizer and model loaded successfully.


In [136]:
def generate_embeddings(text_list):
    """Generates BERT embeddings for a list of text descriptions."""
    encoded_input = tokenizer(text_list, padding=True, truncation=True, return_tensors='pt', max_length=128)
    with torch.no_grad():
        model_output = model(**encoded_input)
    # Use the mean of the token embeddings as the sentence embedding
    embeddings = model_output.last_hidden_state.mean(dim=1)
    return embeddings.tolist()

In [137]:
po_data['po_embeddings'] = generate_embeddings(po_data['cleaned_PO_Description'].tolist())
co_data['co_embeddings'] = generate_embeddings(co_data['cleaned_CO_Description'].tolist())

In [138]:
po_data

,PO,PO_Description,cleaned_PO_Description,po_embeddings
0,PO1,"Apply knowledge of computing, science, and mat...",apply knowledge of computing science and mathe...,"[-0.12511798739433289, 0.2664189338684082, 0.1..."
1,PO2,Use current best practices and standards in so...,use current best practices and standards in so...,"[-0.13032016158103943, 0.3016093969345093, 0.2..."
2,PO3,Analyze complex computing/IT - related problem...,analyze complex computingit related problems ...,"[-0.36303243041038513, 0.22579720616340637, 0...."
3,PO4,Identify and analyze user needs and take them ...,identify and analyze user needs and take them ...,"[-0.00958832073956728, 0.3067125380039215, -0...."
4,PO5,"Design creatively, implement and evaluate diff...",design creatively implement and evaluate diffe...,"[0.19098515808582306, 0.4864211976528168, 0.24..."
5,PO6,Integrate effectively the IT-based solutions i...,integrate effectively the itbased solutions in...,"[-0.005153544247150421, 0.30746299028396606, 0..."
6,PO7,"Select, adapt and apply appropriate techniques...",select adapt and apply appropriate techniques ...,"[-0.30087339878082275, 0.3388974368572235, 0.2..."
7,PO8,"Function effectively as individual, or work co...",function effectively as individual or work col...,"[-0.12778295576572418, 0.2799076437950134, 0.0..."
8,PO9,Assist in the creation of an effective IT proj...,assist in the creation of an effective it proj...,"[-0.1466013342142105, -0.20777928829193115, 0...."
9,PO10,Communicate effectively in both oral and in wr...,communicate effectively in both oral and in wr...,"[-0.11049763113260269, 0.4195915162563324, 0.2..."


In [139]:
po_embeddings_array = np.array(po_data['po_embeddings'].tolist())
co_embeddings_array = np.array(co_data['co_embeddings'].tolist())

In [140]:
similarity_matrix = cosine_similarity(po_embeddings_array, co_embeddings_array)

In [141]:
similarity_matrix

array([[0.7096003 , 0.83247085, 0.71921982, 0.77459634],
       [0.69978793, 0.86710558, 0.79940891, 0.79581719],
       [0.69434023, 0.80705621, 0.7833133 , 0.75944256],
       [0.78359324, 0.84371887, 0.81670348, 0.81446238],
       [0.7126922 , 0.81561881, 0.78839616, 0.75348306],
       [0.71072603, 0.78884279, 0.7379554 , 0.72949022],
       [0.66006797, 0.81705964, 0.73729063, 0.74861366],
       [0.6475271 , 0.70749771, 0.59667293, 0.68628194],
       [0.53231701, 0.76359487, 0.76114487, 0.8813735 ],
       [0.68019148, 0.68799245, 0.56794299, 0.62730778],
       [0.67995125, 0.80669422, 0.7375126 , 0.7687946 ],
       [0.63766079, 0.72308343, 0.74101688, 0.74091118],
       [0.80748355, 0.68189472, 0.54423381, 0.64311079],
       [0.67392822, 0.66948211, 0.58094717, 0.6844049 ],
       [0.47681001, 0.62689608, 0.6209946 , 0.70476193]])

In [142]:
similarity_threshold = 0.7
relationships_df = pd.DataFrame(index=co_data['CO'], columns=po_data['PO'])

In [143]:
for i in range(similarity_matrix.shape[0]): # Iterate through rows (POs)
    for j in range(similarity_matrix.shape[1]): # Iterate through columns (COs)
        po = po_data.loc[i, 'PO']
        co = co_data.loc[j, 'CO']
        # Check if similarity is above the threshold and store as 1 or 0
        relationships_df.loc[co, po] = 1 if similarity_matrix[i, j] >= similarity_threshold else 0

In [144]:
relationships_df

PO,PO1,PO2,PO3,PO4,PO5,PO6,PO7,PO8,PO9,PO10,PO11,PO12,PO13,PO14,PO15
CO,,,,,,,,,,,,,,,
CO1,1,0,0,1,1,1,0,0,0,0,0,0,1,0,0
CO2,1,1,1,1,1,1,1,1,1,0,1,1,0,0,0
CO3,1,1,1,1,1,1,1,0,1,0,1,1,0,0,0
CO4,1,1,1,1,1,1,1,0,1,0,1,1,0,0,1


In [145]:
human_mapping_df = pd.read_csv(HUMAN_MAPPING_FILE)
human_mapping_df = human_mapping_df.set_index('CO')

In [146]:
common_cols = list(set(relationships_df.columns) & set(human_mapping_df.columns))
common_rows = list(set(relationships_df.index) & set(human_mapping_df.index))

relationships_df_aligned = relationships_df.loc[common_rows, common_cols]
human_mapping_df_aligned = human_mapping_df.loc[common_rows, common_cols]

In [147]:
human_mapping_df_aligned = human_mapping_df_aligned.replace(['I', 'D', 'E'], 1) # Assuming 'I', 'D', 'E' indicate a relationship
human_mapping_df_aligned = human_mapping_df_aligned.fillna(0).astype(int) # Replace any remaining NaN with 0 and convert to int
relationships_df_aligned = relationships_df_aligned.astype(int)

C:\Users\Jestoni Andales\AppData\Local\Temp\ipykernel_9192\1337929133.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  human_mapping_df_aligned = human_mapping_df_aligned.replace(['I', 'D', 'E'], 1) # Assuming 'I', 'D', 'E' indicate a relationship


In [148]:
comparison_result = (relationships_df_aligned == human_mapping_df_aligned)

In [149]:
comparison_result

PO,PO12,PO3,PO15,PO1,PO7,PO10,PO9,PO4,PO2,PO8,PO11,PO6,PO5,PO13,PO14
CO,,,,,,,,,,,,,,,
CO3,True,True,False,True,True,False,True,True,True,False,True,True,True,False,False
CO2,True,True,False,True,True,False,True,True,True,True,True,True,True,False,False
CO1,False,False,False,True,False,False,False,True,False,False,False,True,True,True,False
CO4,True,True,True,True,True,False,True,True,True,False,True,True,True,False,False


In [150]:
accuracy = np.sum(comparison_result.values) / comparison_result.size

In [151]:
print(f"Accuracy of BERT mapping compared to human mapping: {accuracy:.2f}\n")
print(f"Human Mapping:")
human_mapping_df_aligned

Accuracy of BERT mapping compared to human mapping: 0.62

Human Mapping:


,PO12,PO3,PO15,PO1,PO7,PO10,PO9,PO4,PO2,PO8,PO11,PO6,PO5,PO13,PO14
CO,,,,,,,,,,,,,,,
CO3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
CO2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
CO1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
CO4,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [152]:
print(f"BERT NLP Mapping:")
relationships_df_aligned

BERT NLP Mapping:


PO,PO12,PO3,PO15,PO1,PO7,PO10,PO9,PO4,PO2,PO8,PO11,PO6,PO5,PO13,PO14
CO,,,,,,,,,,,,,,,
CO3,1,1,0,1,1,0,1,1,1,0,1,1,1,0,0
CO2,1,1,0,1,1,0,1,1,1,1,1,1,1,0,0
CO1,0,0,0,1,0,0,0,1,0,0,0,1,1,1,0
CO4,1,1,1,1,1,0,1,1,1,0,1,1,1,0,0
